In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, cross_val_score, KFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, VotingRegressor
from sklearn.metrics import mean_squared_error
from sklearn.metrics import cohen_kappa_score, make_scorer



In [2]:
def qwk(y_true, y_pred):
    # y_pred vem contínuo (regressão); vamos arredondar
    y_pred_round = np.round(y_pred).astype(int)
    # opcional: garantir que fique no intervalo das classes reais
    y_pred_round = np.clip(y_pred_round, y_true.min(), y_true.max())
    return cohen_kappa_score(y_true, y_pred_round, weights="quadratic")

qwk_scorer = make_scorer(qwk, greater_is_better=True)

In [3]:
# 1) CARREGAR O DATASET
df = pd.read_csv("archive/train.csv")
print(df.columns)

target_col = "quality"

X = df.drop(columns=[target_col, "id"])
y = df[target_col]
df.isnull().sum()



Index(['id', 'fixed acidity', 'volatile acidity', 'citric acid',
       'residual sugar', 'chlorides', 'free sulfur dioxide',
       'total sulfur dioxide', 'density', 'pH', 'sulphates', 'alcohol',
       'quality'],
      dtype='object')


id                      0
fixed acidity           0
volatile acidity        0
citric acid             0
residual sugar          0
chlorides               0
free sulfur dioxide     0
total sulfur dioxide    0
density                 0
pH                      0
sulphates               0
alcohol                 0
quality                 0
dtype: int64

In [4]:
# 2) SPLIT TREINO/TESTE
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [5]:
# 3) DEFINIR MODELOS DE REGRESSÃO

# Modelo 1: Ridge Regression (linear com regularização)
ridge_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Ridge())
])

# Modelo 2: Random Forest Regressor
rf_pipeline = Pipeline([
    ("model", RandomForestRegressor(random_state=42))
])

# Modelo 3: Gradient Boosting Regressor
gb_pipeline = Pipeline([
    ("model", GradientBoostingRegressor(random_state=42))
])


In [6]:
# 4) GRIDS DE HIPERPARÂMETROS

param_grid_ridge = {
    "model__alpha": [0.01, 0.1, 1.0, 10.0, 100.0]
}

param_grid_rf = {
    "model__n_estimators": [200, 400, 600],
    "model__max_depth": [None, 8, 12, 16],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4],
    "model__max_features": ["sqrt", "log2"]
}


param_grid_gb = {
    "model__n_estimators": [100, 300, 500],
    "model__learning_rate": [0.03, 0.05, 0.1],
    "model__max_depth": [2, 3, 4],
    "model__subsample": [0.8, 1.0],
    "model__min_samples_leaf": [1, 3, 5]
}



In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)

# 5) GRID SEARCH PARA CADA MODELO

grid_ridge = GridSearchCV(
    estimator=ridge_pipeline,
    param_grid=param_grid_ridge,
    cv=cv,
    scoring=qwk_scorer,
    n_jobs=-1
)

grid_rf = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=param_grid_rf,
    cv=cv,
    scoring=qwk_scorer,
    n_jobs=-1
)

grid_gb = GridSearchCV(
    estimator=gb_pipeline,
    param_grid=param_grid_gb,
    cv=cv,
    scoring=qwk_scorer,
    n_jobs=-1
)

# Treinar (isso pode demorar um pouquinho)
grid_ridge.fit(X_train, y_train)
grid_rf.fit(X_train, y_train)
grid_gb.fit(X_train, y_train)


In [ ]:
print("Ridge - melhores parâmetros:", grid_ridge.best_params_)
print("Ridge - melhor QWK (CV):",    grid_ridge.best_score_)

print("Random Forest - melhores parâmetros:", grid_rf.best_params_)
print("Random Forest - melhor QWK (CV):",    grid_rf.best_score_)

print("Gradient Boosting - melhores parâmetros:", grid_gb.best_params_)
print("Gradient Boosting - melhor QWK (CV):", grid_gb.best_score_)


In [ ]:
# 6) VOTING REGRESSOR COM OS 3 MODELOS TUNADOS

voting_reg = VotingRegressor(
    estimators=[
        ("ridge", grid_ridge.best_estimator_),
        ("rf", grid_rf.best_estimator_),
        ("gb", grid_gb.best_estimator_)
    ]
)

# Avaliar com cross-validation no conjunto completo
scores_qwk = cross_val_score(
    voting_reg,
    X, y,
    cv=cv,
    scoring=qwk_scorer,
    n_jobs=-1
)

print("VotingRegressor - QWK médio (CV):", np.mean(scores_qwk))
print("VotingRegressor - QWK desvio padrão:", np.std(scores_qwk))


In [ ]:
from sklearn.metrics import mean_squared_error

voting_reg.fit(X_train, y_train)
y_pred_voting = voting_reg.predict(X_test)

# RMSE só se quiser comparar com antes
rmse_voting_test = np.sqrt(mean_squared_error(y_test, y_pred_voting))
print("VotingRegressor - RMSE no teste:", rmse_voting_test)

# QWK no teste
y_pred_round = np.round(y_pred_voting).astype(int)
y_pred_round = np.clip(y_pred_round, y_test.min(), y_test.max())

qwk_test = cohen_kappa_score(y_test, y_pred_round, weights="quadratic")
print("VotingRegressor - QWK no teste:", qwk_test)


In [ ]:
# Refitar o VotingRegressor em TODO o dataset de treino
voting_reg.fit(X, y)


In [ ]:
df_test = pd.read_csv("archive/test.csv")
df_test.head()

In [ ]:
X_kaggle_test = df_test.drop(columns=["id"])


In [ ]:
# Prever valores contínuos
y_pred_test = voting_reg.predict(X_kaggle_test)

# Arredondar para o inteiro mais próximo
y_pred_round = np.round(y_pred_test).astype(int)

# Opcional: garantir que fique dentro do intervalo válido [3, 8]
y_pred_round = np.clip(y_pred_round, 3, 8)


In [ ]:
submission = pd.DataFrame({
    "id": df_test["id"],
    "quality": y_pred_round
})

submission.head()


In [ ]:
submission.to_csv("submission.csv", index=False)
print("Arquivo submission.csv criado!")


In [ ]:
print("Valores únicos previstos:", np.unique(y_pred_round))
print("Mínimo:", y_pred_round.min(), "Máximo:", y_pred_round.max())
